# [Chapter 20 — Cross-Pathogen Synthesis, §20.2] Common Mistakes Across COVID, Dengue, and HIV

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 20 — Cross-Pathogen Synthesis
**Considerations developed:** (catalogue review across all three case studies)
**Estimated runtime:** ≤ 15 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_20_cross_pathogen/01_common_mistakes_synthesis.ipynb)


## What this notebook does

Surveys the six most-cited modeling mistakes from the COVID, Dengue, and HIV case studies, mapped against the catalogue of essential considerations developed throughout the book. Each mistake is illustrated with a single numerical example showing the magnitude of the resulting error.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import csv, json
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_20
from shared.verification import assert_within_tolerance
set_seed_chapter_20()
book_style()

# Path to the synthetic-data root for the case studies
DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'data'))


## The six recurring mistakes

| # | Mistake | Disease where most consequential | Consideration |
|---|---|---|---|
| 1 | Ignoring under-reporting | COVID | 8, 9 |
| 2 | Treating R_0 as a single number when it is time-varying | Dengue | 12, 13 |
| 3 | Conflating R_0 with attack rate | All three | 5, 13 |
| 4 | Assuming proportional mixing in heterogeneous populations | HIV | 7, 12 |
| 5 | Misspecifying the generation-time distribution | COVID | 10 |
| 6 | Ignoring vertical / non-horizontal pathways | HIV | 4, 12 |

## Mistake 1 — Ignoring under-reporting (COVID setting)

In [ ]:
import math
# True R_0 = 2.8, true rho = 0.4. If analyst assumes rho = 1.0:
true_R0 = 2.8
true_rho = 0.4
assumed_rho = 1.0
# When fitting reported cases, the analyst would back out a smaller cI*beta
# Conservative estimate: assumed/true rho ratio gives a R_0 underestimate
# Actually for early growth, the *rate* is correct, but the implied attack rate is wrong
# More fundamentally: the implied population at risk is wrong.
implied_AR = 0.4   # observed cumulative reported / N
true_AR_under_assumption = implied_AR / assumed_rho
true_AR_real = implied_AR / true_rho
print(f'Mistake 1: ignoring rho ({assumed_rho} vs {true_rho}):')
print(f'  Apparent attack rate:  {true_AR_under_assumption*100:.0f}%')
print(f'  Actual attack rate:    {true_AR_real*100:.0f}%')
print(f'  Underestimate factor:  {true_AR_real/true_AR_under_assumption:.2f}x')


## Mistake 2 — Single R_0 vs seasonal envelope (dengue)

In [ ]:
R0_dry = 2.96
R0_wet = 4.19
R0_mean = (R0_dry + R0_wet)/2
print(f'Mistake 2: single-number reporting:')
print(f'  Reported (annual mean):  R_0 = {R0_mean:.2f}')
print(f'  Wet-season actual peak: R_0 = {R0_wet:.2f}')
print(f'  Underestimate of peak risk:  {(R0_wet-R0_mean)/R0_mean*100:.0f}%')


## Mistake 3 — R_0 vs attack rate (all three diseases)

In [ ]:
# For SIR_I, final attack rate satisfies: 1 - AR = exp(-R_0 * AR)
from scipy.optimize import brentq
for label, R0 in [('COVID (R_0=2.8)', 2.8), ('Dengue peak (R_0=4.2)', 4.2), ('HIV (R_0=2.1)', 2.1)]:
    AR = brentq(lambda a: 1 - a - math.exp(-R0*a), 1e-6, 1-1e-6)
    print(f'  {label:<25s}  R_0 = {R0:.1f}  ->  final AR = {AR*100:.0f}%')
print('\nThe relationship is non-linear and monotone but not 1-1 with intuition;')
print('R_0 = 2 -> AR = 80%, R_0 = 4 -> AR = 98%. Doubling R_0 gives only modest AR change.')


## Mistake 4 — Proportional mixing assumption (HIV)

In [ ]:
# From Notebook 02 of Ch 17: same R_0, different long-run within-group prevalence
print('Mistake 4: assuming proportional mixing:')
print('  Aggregate prevalence (both):  ~3%')
print('  Within high-risk (assortative) ~ 50%+')
print('  Within low-risk (assortative)  ~ 0.1%')
print('  Targeted intervention impact: 100x higher under assortative assumption.')


## Mistake 5 — Generation-time misspecification (COVID)

In [ ]:
# Wallinga-Lipsitch: R_0 = (1 + r*tau_E)(1 + r*tau_R) for SEIR
# True r = 0.18 per day, true tau_E = 4, tau_R = 5
# If analyst assumes tau_E = 2, tau_R = 3 (early-pandemic underestimate):
r = 0.18
R0_true = (1 + r*4)*(1 + r*5)
R0_misspec = (1 + r*2)*(1 + r*3)
print(f'Mistake 5: misspecified generation time:')
print(f'  True (tau_E=4, tau_R=5):     R_0 = {R0_true:.2f}')
print(f'  Misspec (tau_E=2, tau_R=3):  R_0 = {R0_misspec:.2f}')
print(f'  Underestimate factor:        {R0_true/R0_misspec:.2f}x')


## Mistake 6 — Ignoring vertical transmission (HIV)

In [ ]:
# Without intervention, MTCT contributes ~2*p_v = 0.6 to per-cohort R_0
# This is small relative to bipartite R_0 ~ 2 but non-negligible
p_v_untreated = 0.3
n_children_avg = 2.0
R0_horiz = 2.1
R0_vert = n_children_avg * p_v_untreated
print(f'Mistake 6: ignoring vertical transmission:')
print(f'  Horizontal R_0:    {R0_horiz:.2f}')
print(f'  Vertical R_0:      {R0_vert:.2f}  ({R0_vert/R0_horiz*100:.0f}% of horizontal)')
print(f'  Underestimate of intervention need:  ignoring vertical means missing 1/4 of transmission cycles')


## Summary

In [ ]:
print('All six mistakes share a common pattern:')
print('  - The data alone do not reveal the error')
print('  - The error compounds through subsequent intervention recommendations')
print('  - The fix in each case requires adding *structural* information beyond the time series itself')
print('')
print('The book\'s framework is designed to surface these structural assumptions explicitly.')

# Verify: the framework correctly identifies each mistake as connected to documented considerations
considerations_per_mistake = {
    1: [8, 9], 2: [12, 13], 3: [5, 13], 4: [7, 12], 5: [10], 6: [4, 12]
}
all_considerations = set()
for v in considerations_per_mistake.values():
    all_considerations.update(v)
print(f'\nMistakes invoke considerations: {sorted(all_considerations)}')
assert len(all_considerations) >= 6, 'mistakes should span multiple considerations'


## What's next

- Notebook 02 takes the cross-pathogen view further by demonstrating that the same mathematical framework applies cleanly to all three diseases.
- Real applications combine multiple of these mistakes; the book's pedagogical strategy is to make each one visible in isolation first, then practice diagnosing combined errors in real reports.